# Tests: `fasterai.quantize.fake_quantize_callback` (source `nbs/quantize/fake_quantize_callback.ipynb`)

In [ ]:
from fastcore.test import *
import contextlib, copy, io, os, random, re, tempfile
from pathlib import Path
import torch
import torch.nn as nn
from fastai.callback.core import Callback
from fastai.callback.tracker import SaveModelCallback
from fastai.data.core import DataLoaders
from fastai.learner import Learner
from torch.utils.data import TensorDataset
from fasterai.core.criteria import large_final
from fasterai.core.parametrize import _is_parametrized, _master
from fasterai.core.precision import FakeQuantSpec, fake_quant_spec
from fasterai.core.schedule import one_shot
from fasterai.prune.pruner import Pruner
from fasterai.quantize.fake_quantize_callback import *
from fasterai.quantize.fake_quantize_callback import _ActObserver, _FakeQuantWeight
from fasterai.quantize.fake_quantizer import FakeQuantizer, _ActRounder, _fake_quantize
from fasterai.quantize.quantize_callback import QuantizeCallback
from fasterai.sparse.sparsifier import Sparsifier
from fasterai.sparse.sparsify_callback import SparsifyCallback

In [ ]:
def _model():
    "The model every fit below trains: two convolutions and a Linear, all rounded by default"
    torch.manual_seed(0)
    return nn.Sequential(nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(),
                         nn.Conv2d(8, 4, 3, padding=1), nn.AdaptiveAvgPool2d(1),
                         nn.Flatten(), nn.Linear(4, 2))

def _bn_model():
    "The same with a BatchNorm: `Sparsifier`'s has-weight loop then sees a module nothing parametrizes"
    torch.manual_seed(0)
    return nn.Sequential(nn.Conv2d(3, 8, 3, padding=1), nn.BatchNorm2d(8), nn.ReLU(),
                         nn.Conv2d(8, 4, 3, padding=1), nn.AdaptiveAvgPool2d(1),
                         nn.Flatten(), nn.Linear(4, 2))

def _dls(n=32, bs=8):
    "A fastai `DataLoaders` over random images: 32 items at bs 8 is a 4-step fit"
    # a fastai loader draws its shuffling seed from Python's `random` at construction, so two fits are
    # only comparable when that one is seeded too
    torch.manual_seed(0); random.seed(0)
    ds = TensorDataset(torch.randn(n, 3, 8, 8), torch.randint(0, 2, (n,)))
    return DataLoaders.from_dsets(ds, ds, bs=bs, device='cpu')

_VALID_SCALE = 1000.   # the validation batches are this many times louder than the training ones

def _ramp_dls(n=32, bs=8):
    "Training batches whose range grows, in order, against validation batches nothing may observe"
    train = torch.stack([torch.full((3, 8, 8), 1. + i) for i in range(n)])
    valid = torch.full((n, 3, 8, 8), _VALID_SCALE)
    zeros = torch.zeros(n, dtype=torch.long)
    return DataLoaders.from_dsets(TensorDataset(train, zeros), TensorDataset(valid, zeros),
                                  bs=bs, device='cpu', shuffle=False)

def _learn(model=None, dls=None, **kwargs):
    return Learner(dls or _dls(), model or _model(), loss_func=nn.CrossEntropyLoss(), **kwargs)

def _fit(learn, cbs, n=1):
    "One fit with the callbacks' printing captured — the cells here read the model, not the log"
    with contextlib.redirect_stdout(io.StringIO()) as out:
        learn.fit(n, cbs=cbs)
    return out.getvalue()

def _install(model, learn=None, **kwargs):
    "Install the callback on `model` without running a fit, exactly as `before_fit` does"
    cb = FakeQuantizeCallback(model=model, **kwargs)
    cb.learn = learn if learn is not None else _learn(model)
    with contextlib.redirect_stdout(io.StringIO()): cb.before_fit()
    return cb

def _zeros_pct(m): return (_master(m) == 0).float().mean().item()

# the fixture really is deterministic, so two arms of a golden are comparable
test_eq(torch.equal(_model()[0].weight, _model()[0].weight), True)

## The rounding is the post-training one

In [ ]:
# --- GOLDEN: the training-time rounding IS the post-training one, computed live in this run ---
# Both arms are built here, from the same tensor and the same seed: there is no stored digest to drift
# against, and any divergence between the two paths shows up as different floating-point bytes.
torch.manual_seed(0)
_w = torch.nn.Parameter(torch.randn(8, 12))
for _bits, _sym, _axis, _gs in ((8, True, 'per_channel', None), (4, True, 'per_tensor', None),
                                (4, False, 'per_channel', None), (3, True, 'per_group', 4),
                                (2, True, 'per_channel', None)):
    _rounded = _FakeQuantWeight(_bits, _sym, _axis, _gs)(_w)
    with torch.no_grad(): _ptq = _fake_quantize(_w.detach(), _bits, _sym, _axis, _gs)
    test_eq(torch.equal(_rounded.detach(), _ptq), True)
    test_ne(_rounded.detach().tolist(), _w.detach().tolist())   # ...and it is not a pass-through

# the same claim on a whole model: a fit of zero steps installs and bakes, and must land byte for byte
# where `FakeQuantizer.quantize_model()` lands
_m_qat, _m_ptq = _model(), _model()
_fit(_learn(_m_qat), [FakeQuantizeCallback(weight_bits=4)], n=0)
FakeQuantizer(_m_ptq, 4).quantize_model()
_ptq_params = dict(_m_ptq.named_parameters())
test_eq(sorted(dict(_m_qat.named_parameters())), sorted(_ptq_params))
for _n, _a in _m_qat.named_parameters():
    test_eq((_n, torch.equal(_a, _ptq_params[_n])), (_n, True))
# the buffers too, so the bake is pinned to leave no observation state of its own behind
_ptq_buffers = dict(_m_ptq.named_buffers())
test_eq(sorted(dict(_m_qat.named_buffers())), sorted(_ptq_buffers))
assert _ptq_buffers, 'neither arm holds a buffer: this check proves nothing'
for _n, _b in _m_qat.named_buffers():
    test_eq((_n, torch.equal(_b, _ptq_buffers[_n])), (_n, True))
test_eq(fake_quant_spec(_m_ptq).trained, False)   # and only one of the two claims it trained
test_eq(fake_quant_spec(_m_qat).trained, True)

## The gradient

In [ ]:
# --- the gradient: straight through to the master, and clipped by the frozen activation grid ---
# on the parametrization itself the estimator is the identity: d(rounded weight)/d(master) is 1
_w = torch.randn(4, 6, requires_grad=True)
_FakeQuantWeight(4, True, 'per_channel', None)(_w).sum().backward()
test_eq(torch.equal(_w.grad, torch.ones_like(_w)), True)

_m = _model()
_cb = _install(_m, weight_bits=4)
test_eq(_m[0].weight.is_leaf, False)     # `m.weight` is computed, so it carries no gradient of its own
_x = torch.randn(2, 3, 8, 8)
_m(_x).sum().backward()

# so what the master receives is exactly what the ROUNDED weight received: a model holding the same
# rounded weights outright, with nothing parametrized, produces the same gradients
_ref = _model()
FakeQuantizer(_ref, 4).quantize_model()
_ref(_x).sum().backward()
for _i in (0, 2, 5):
    test_eq((_i, torch.equal(_master(_m[_i]).grad, _ref[_i].weight.grad)), (_i, True))
assert float(_master(_m[0]).grad.abs().sum()) > 0, 'no gradient reached the master at all'

# the ACTIVATION grid is frozen, and there the estimator really does clip. The probe is the identity,
# so the gradient reaching its input is the one the rounding let through.
_probe = nn.Linear(4, 4, bias=False)
with torch.no_grad(): _probe.weight.copy_(torch.eye(4))
_obs = _ActObserver(8, True, None)
_h = _probe.register_forward_hook(_obs)
_probe.train()
_probe(torch.tensor([[-1., -0.5, 0.5, 1.]]))    # observe the range [-1, 1]
_obs.frozen = True
_x = torch.tensor([[-2., 0., 0.5, 2.]], requires_grad=True)
_probe(_x).sum().backward()
test_eq(_x.grad.tolist(), [[0., 1., 1., 0.]])
_h.remove()

In [ ]:
# --- the optimizer `fit` builds BEFORE the callback runs still trains the right tensor ---
_m = _model()
_opt = torch.optim.SGD(_m.parameters(), lr=0.5)
_held = {id(p) for _group in _opt.param_groups for p in _group['params']}
_cb = _install(_m, weight_bits=4)
_id_master = id(_master(_m[0]))
assert _id_master in _held, 'the parametrization replaced the parameter the optimizer holds'
_before = _master(_m[0]).detach().clone()
_m(torch.randn(2, 3, 8, 8)).sum().backward()
_opt.step()
test_ne(_master(_m[0]).detach().tolist(), _before.tolist())
# ...and the bake hands the same object back, so nothing has to be rebound around it either
_cb.bake()
test_eq(id(_m[0].weight), _id_master)
test_eq(isinstance(_m[0].weight, nn.Parameter), True)

## What a fit leaves behind

In [ ]:
# --- what a fit leaves behind: an ordinary module, and the trained master one call away ---
_m = _model()
_keys, _sd0 = list(_m.state_dict()), copy.deepcopy(_m.state_dict())
_w0 = _m[0].weight.detach().clone()
_cb = FakeQuantizeCallback(4, 8)
_out = _fit(_learn(_m), [_cb])
assert 'Training through W4A8' in _out, _out

for _mod in (_m[0], _m[2], _m[5]):
    test_eq(_is_parametrized(_mod), False)
# the same state dict keys (the removal reorders them), and the pre-fit checkpoint still loads
test_eq(sorted(_m.state_dict()), sorted(_keys))
test_eq(set(_m.state_dict()), set(_sd0))
_m.load_state_dict(_sd0)
torch.save(_m, io.BytesIO())    # a parametrized model cannot be serialized; a baked one can

# every layer holds the rounding of ITS trained master, and the fit really moved that master
_m = _model()
_cb = FakeQuantizeCallback(4)
_fit(_learn(_m), [_cb])
_trained = _m[0]._fp_weight.detach().clone()
test_eq(torch.equal(_m[0].weight.detach(), _fake_quantize(_trained, 4, True, 'per_channel')), True)
test_ne(_trained.tolist(), _w0.tolist())

# remove() gives back the TRAINED master, not the weights the fit started from
_cb.fake_quantizer.remove()
test_eq(torch.equal(_m[0].weight.detach(), _trained), True)
test_ne(_m[0].weight.detach().tolist(), _w0.tolist())
test_eq(fake_quant_spec(_m), None)
test_eq([_n for _n, _ in _m.named_buffers()], [])

In [ ]:
# --- a second fit (what `fine_tune` runs) resumes from the master, and re-snapshots it ---
_m = _model()
_learn_two = _learn(_m)
_cb = FakeQuantizeCallback(weight_bits=4)
_fit(_learn_two, [_cb])
_first = _m[0]._fp_weight.detach().clone()
_fit(_learn_two, [_cb])
_second = _m[0]._fp_weight.detach().clone()
test_ne(_second.tolist(), _first.tolist())      # the snapshot is the second fit's master, not the first's
_cb.fake_quantizer.remove()
test_eq(torch.equal(_m[0].weight.detach(), _second), True)
test_eq(fake_quant_spec(_m), None)

## The activation scales

In [ ]:
# --- the freeze, on the 4-iteration fit every CPU fixture in this repo is ---
# NOTE: `pct_train` is incremented AFTER each batch, so the largest value a fit of N steps ever
# shows inside `before_batch` is (N-1)/N — 0.75 here. A threshold of 0.9 is never reached, which is why
# `bake()` freezes unconditionally as well.
class _Watch(Callback):
    "Records what the callback under test had done by the time each training batch started"
    order = 71   # just above FakeQuantizeCallback, so it sees the freeze of the same batch
    def __init__(self, cb): self.cb, self.seen, self.scales = cb, [], []
    def before_batch(self):
        if self.training: self.seen.append((round(self.pct_train, 3), self.cb.frozen))
    def after_batch(self):
        if self.training: self.scales.append(float(self.cb.fake_quantizer.model[0]._act_scale))

_arms = {}
for _pct in (0.5, 1.0):
    _m = _model()
    _cb = FakeQuantizeCallback(8, 8, freeze_act_pct=_pct, act_averaging=None)
    _watch = _Watch(_cb)
    _fit(_learn(_m, dls=_ramp_dls()), [_cb, _watch])
    _arms[_pct] = (_m, _cb, _watch)

# the threshold that IS reached fires on the batch it names, and the one that is not never fires
test_eq([_p for _p, _ in _arms[0.5][2].seen], [0.0, 0.25, 0.5, 0.75])
test_eq([_f for _, _f in _arms[0.5][2].seen], [False, False, True, True])
test_eq([_f for _, _f in _arms[1.0][2].seen], [False, False, False, False])

# ...and freezing is not a flag: the scales stop moving, on data whose range keeps growing. The freeze
# happens in `before_batch`, so the scale recorded after batch 1 is the last one the fit observed.
_early, _late = _arms[0.5][2].scales, _arms[1.0][2].scales
assert _early[0] < _early[1], _early
test_eq(_early[1:], [_early[1]] * 3)
assert all(_a < _b for _a, _b in zip(_late, _late[1:])), _late   # nothing froze this arm during the fit
assert _late[-1] > _early[-1], (_late, _early)

# both models leave the fit frozen: the fit that never reached its threshold was frozen by the bake
for _pct, (_m, _cb, _watch) in _arms.items():
    test_eq((_pct, _cb.frozen), (_pct, True))
    test_eq((_pct, isinstance(next(iter(_cb.fake_quantizer._act_hooks.values()))[1], _ActRounder)),
            (_pct, True))
    test_eq((_pct, float(_m[0]._act_scale)), (_pct, _watch.scales[-1]))   # the bake kept what was observed

In [ ]:
# --- the observer: a moving average by default, and blind to validation ---
_probe = nn.Linear(4, 4, bias=False)
with torch.no_grad(): _probe.weight.copy_(torch.eye(4))

for _averaging, _expected in ((None, 10.0), (0.5, 5.5), (0.01, 1.09)):
    _p = copy.deepcopy(_probe).train()
    _p.register_forward_hook(_ActObserver(8, True, _averaging))
    _p(torch.ones(1, 4))            # the range starts at [0, 1]
    _p(torch.full((1, 4), 10.))     # one loud batch: absolute min/max jumps to it, an average creeps
    test_close(float(_p._act_max), _expected, eps=1e-5)
    # an eval() batch never moves it, however loud — a plain hook fires during validation too
    _p.eval()
    _p(torch.full((1, 4), 1000.))
    test_close(float(_p._act_max), _expected, eps=1e-5)
    # ...and it is still rounded in eval, onto the grid training left
    assert float(_p(torch.full((1, 4), 1000.)).max()) < 1000., 'the frozen grid did not clip'

# the same claim through a fit: the validation batches are 1000x the training ones, and the grid the
# model leaves with must stay below what a single validation batch would have shown
_m = _model()
_cb = FakeQuantizeCallback(8, 8, freeze_act_pct=1.0, act_averaging=None)
_fit(_learn(_m, dls=_ramp_dls()), [_cb])
for _, _rounder in _cb.fake_quantizer._act_hooks.values(): _rounder.enabled = False
with torch.no_grad(): _valid_max = _m[0](torch.full((2, 3, 8, 8), _VALID_SCALE)).abs().max().item()
for _, _rounder in _cb.fake_quantizer._act_hooks.values(): _rounder.enabled = True
_covered = float(_m[0]._act_scale) * 127     # a symmetric 8-bit grid reaches 127 steps
assert 0 < _covered < _valid_max / 10, (_covered, _valid_max)
# the bake leaves exactly the post-training state: the observation buffers are gone
test_eq([_n for _n, _ in _m.named_buffers() if _n.endswith(('_act_min', '_act_max'))], [])

In [ ]:
# --- `self.fake_quantizer` is public, and `calibrate` must observe UNROUNDED activations mid-QAT ---
# It stands the rounding down through the `enabled` flag every hook of this module honors. An observer
# that ignored it would measure its OWN output, clipped onto the grid the fit had so far, and freeze a
# scale off by the ratio between the two ranges.
def _dls_of(x):
    "A fastai `DataLoaders` over one batch of `x` — the calibration input `FakeQuantizer` accepts"
    _ds = TensorDataset(x, torch.zeros(len(x), dtype=torch.long))
    return DataLoaders.from_dsets(_ds, _ds, bs=len(x), device='cpu')

_m = _model()
_cb = _install(_m, weight_bits=8, act_bits=4, act_averaging=None)   # 4 bits, so the rounding really bites
_quiet, _loud = torch.full((8, 3, 8, 8), 0.05), torch.full((8, 3, 8, 8), 5.)
_m.train()
_m(_quiet)                        # one training batch: the fit's grid is the quiet one
_narrow = float(_m[0]._act_scale)
_m.eval()

def _unrounded_scale(x, bits=4):
    "The range this layer's output really spans, over the largest integer of a symmetric grid"
    for _, _r in _cb.fake_quantizer._act_hooks.values(): _r.enabled = False
    with torch.no_grad(): _out = _m[0](x)
    for _, _r in _cb.fake_quantizer._act_hooks.values(): _r.enabled = True
    return max(_out.max().item(), -_out.min().item()) / (2 ** (bits - 1) - 1)

_expected = _unrounded_scale(_loud)
_cb.fake_quantizer.calibrate(_dls_of(_loud))
test_close(float(_m[0]._act_scale), _expected, eps=1e-6)
assert float(_m[0]._act_scale) > 5 * _narrow, (float(_m[0]._act_scale), _narrow)
# the hooks are handed back rounding, and the model still runs
for _, _r in _cb.fake_quantizer._act_hooks.values(): test_eq(_r.enabled, True)
with torch.no_grad(): test_eq(_m(_loud).shape, (8, 2))

## Inside the training loop

In [ ]:
# --- order 70, and what it is for ---
test_eq(FakeQuantizeCallback.order, 70)
assert FakeQuantizeCallback.order > SaveModelCallback.order, 'a checkpoint would be loaded after the bake'
test_eq(SaveModelCallback.order, 61)

# SaveModelCallback co-exists: its after_fit loads a checkpoint written while the model was
# parametrized, and it must land in a model that still is
_m = _model()
_learn_s = _learn(_m)
_learn_s.path = Path(tempfile.mkdtemp())
_cb = FakeQuantizeCallback(weight_bits=4)
_fit(_learn_s, [SaveModelCallback(fname='qat_ck'), _cb], n=2)
test_eq(_is_parametrized(_m[0]), False)
test_eq(fake_quant_spec(_m).trained, True)
test_eq(torch.equal(_m[0].weight.detach(), _fake_quantize(_m[0]._fp_weight, 4, True, 'per_channel')), True)

In [ ]:
# --- provenance: `trained` is True after a fit and False on every post-training path ---
_m = _model()
_cb = FakeQuantizeCallback(4, 8, symmetric=False, qscheme='per_tensor')
_fit(_learn(_m), [_cb])
_spec = fake_quant_spec(_m)
test_eq((_spec.trained, _spec.label, _spec.qscheme, _spec.symmetric), (True, 'W4A8', 'per_tensor', False))
test_eq(_spec.observer, 'static')
test_eq(_spec, _cb.fake_quantizer.spec.__class__(**{**_cb.fake_quantizer.spec.as_dict(), 'trained': True}))
with ExceptionExpected(AttributeError): _spec.trained = False   # frozen, like every other field

# the quantizer itself never claims a fit happened
test_eq(_cb.fake_quantizer.spec.trained, False)
test_eq(FakeQuantSpec(8, None, 'per_channel', True, 'static').trained, False)
for _make in (lambda m: FakeQuantizer(m, 8),
              lambda m: FakeQuantizer(m, 8, 8, observer='dynamic'),
              lambda m: FakeQuantizer(m, 4, qscheme='per_tensor')):
    _p = _model()
    _make(_p).quantize_model()
    test_eq(fake_quant_spec(_p).trained, False)
_p = _model()
_fq = FakeQuantizer(_p, 8, 8)
_fq.calibrate(_dls())
_fq.quantize_model()
test_eq(fake_quant_spec(_p).trained, False)

In [ ]:
# --- a fit that raises leaves the parametrizations in place, and both ways out work ---
def _boom(pred, targ): raise RuntimeError('metric exploded')

def _interrupted(**kwargs):
    "A model whose fit died inside a metric, and the callback that was installed on it"
    m = _model()
    cb = FakeQuantizeCallback(**kwargs)
    learn = Learner(_dls(), m, loss_func=nn.CrossEntropyLoss(), metrics=_boom)
    test_fail(lambda: _fit(learn, [cb]), contains='metric exploded', exc=RuntimeError)
    return m, cb

_m, _cb = _interrupted(weight_bits=4)
test_eq(_is_parametrized(_m[0]), True)          # `after_fit` never ran
test_eq(fake_quant_spec(_m), None)
test_fail(lambda: torch.save(_m, io.BytesIO()), contains='state_dict', exc=RuntimeError)
# ...and fitting again refuses rather than parametrizing a parametrized model twice
test_fail(lambda: _fit(_learn(_m), [_cb]), contains='still installed', exc=RuntimeError)

# way out 1: strip() — the floating-point weights, as the interrupted fit left them
_master0 = _master(_m[0]).detach().clone()
_cb.strip()
test_eq(_is_parametrized(_m[0]), False)
test_eq(torch.equal(_m[0].weight.detach(), _master0), True)
test_eq(fake_quant_spec(_m), None)
test_eq([_n for _n, _ in _m.named_buffers()], [])
torch.save(_m, io.BytesIO())
test_fail(_cb.bake, contains='Nothing is installed to bake', exc=RuntimeError)   # nothing left

# way out 2: bake() — keep the rounding the interrupted fit was training through
_m, _cb = _interrupted(weight_bits=4, act_bits=8)
_master0 = _master(_m[0]).detach().clone()
_cb.bake()
test_eq(_is_parametrized(_m[0]), False)
test_eq(torch.equal(_m[0].weight.detach(), _fake_quantize(_master0, 4, True, 'per_channel')), True)
test_eq(fake_quant_spec(_m).trained, True)
test_fail(_cb.strip, contains='fake_quantizer.remove()', exc=RuntimeError)
torch.save(_m, io.BytesIO())

In [ ]:
# --- a refused bake is all-or-nothing: nothing is written, and the way out still works ---
# The refusal is at the top of `bake()`, before a single weight: baking halfway would leave the model
# unparametrized with `strip()` promising floating-point weights it could no longer give back.
_m = _model()
_before = [_master(_m[_i]).detach().clone() for _i in (0, 2, 5)]
_cb = _install(_m, weight_bits=4, act_bits=8)     # installed, and no batch has run yet
test_fail(_cb.bake, contains='never saw a training batch', exc=RuntimeError)
for _i in (0, 2, 5): test_eq((_i, _is_parametrized(_m[_i])), (_i, True))
test_eq(fake_quant_spec(_m), None)
test_eq(_cb._installed, True)

# ...so `strip()` gives back exactly the weights the callback was handed
_cb.strip()
for _i, _w in zip((0, 2, 5), _before):
    test_eq((_i, torch.equal(_m[_i].weight.detach(), _w)), (_i, True))

# the same refusal on the user-side path: a fit of zero steps observes no activation either
test_fail(lambda: _fit(_learn(_model()), [FakeQuantizeCallback(8, 8)], n=0),
          contains='never saw a training batch', exc=RuntimeError)
# a weight-only fit of zero steps has nothing to observe, and bakes
_m = _model()
_fit(_learn(_m), [FakeQuantizeCallback(weight_bits=8)], n=0)
test_eq(fake_quant_spec(_m).trained, True)

In [ ]:
# --- learn.save / learn.load round-trip mid-fit, on the parametrized model ---
_m = _model()
_learn_sl = _learn(_m)
_learn_sl.path = Path(tempfile.mkdtemp())
_cb = _install(_m, learn=_learn_sl, weight_bits=4)
assert '0.parametrizations.weight.original' in _m.state_dict(), list(_m.state_dict())
_learn_sl.save('mid')
_master0 = _master(_m[0]).detach().clone()
with torch.no_grad(): _master(_m[0]).mul_(0.)
_learn_sl.load('mid')
test_eq(torch.equal(_master(_m[0]).detach(), _master0), True)
test_eq(_is_parametrized(_m[0]), True)

## What it refuses

In [ ]:
# --- what it refuses, and when ---
# its own arguments are checked at construction, where they are read
test_fail(lambda: FakeQuantizeCallback(freeze_act_pct=1.5), contains='fraction of the training')
test_fail(lambda: FakeQuantizeCallback(freeze_act_pct=-0.1), contains='fraction of the training')
test_fail(lambda: FakeQuantizeCallback(act_averaging=0), contains='averaging constant')
test_fail(lambda: FakeQuantizeCallback(act_averaging=1.5), contains='averaging constant')
test_fail(FakeQuantizeCallback().bake, contains='Nothing is installed to bake', exc=RuntimeError)
test_fail(FakeQuantizeCallback().strip, contains='nothing to strip', exc=RuntimeError)

# the width grammar is `FakeQuantizer`'s, and it refuses at the start of the fit — with the same
# sentences, and before a single weight is touched. Each callback is built OUTSIDE the lambda, so
# this pins WHERE the refusal happens: construction succeeds, and the fit is what raises.
for _kwargs, _message in (({'weight_bits': 1}, '[2, 16]'), ({'act_bits': 17}, '[2, 16]'),
                          ({'qscheme': 'per_axis'}, 'per_channel'), ({'observer': 'ema'}, 'dynamic'),
                          ({'weight_bits': 4, 'qscheme': 'per_group', 'group_size': 8},
                           "the 27 weights of a row of '0'")):
    _m = _model()
    _cb_bad = FakeQuantizeCallback(**_kwargs)
    test_fail(lambda: _fit(_learn(_m), [_cb_bad]), contains=_message)
    test_eq((_message, _is_parametrized(_m[0])), (_message, False))
test_fail(lambda: _install(nn.Sequential(nn.ReLU())), contains='layer_type')

# a weight something else already computes is not silently wrapped a second time
_m = _model()
_cb = _install(_m, weight_bits=8)
_other = FakeQuantizeCallback(model=_m)
_other.learn = _learn(_m)
test_fail(_other.before_fit, contains='already computes its weight', exc=RuntimeError)
test_eq(len(_m[0].parametrizations.weight), 1)

# QuantizeCallback swaps the model out from under these parametrizations: the two are refused together
_learn_two = _learn()
_qcb, _fcb = QuantizeCallback(), FakeQuantizeCallback()
_learn_two.add_cbs([_qcb, _fcb])
test_fail(_fcb.before_fit, contains='keep one')
assert not any(_is_parametrized(_mod) for _mod in _learn_two.model.modules())

In [ ]:
# --- act_bits=None: `freeze_act_pct` is a documented no-op, not a refusal ---
_m = _model()
_cb = FakeQuantizeCallback(weight_bits=8, act_bits=None, freeze_act_pct=0.1)
_out = _fit(_learn(_m), [_cb])
test_eq(fake_quant_spec(_m).label, 'W8AF')
test_eq([_n for _n, _ in _m.named_buffers() if '_act' in _n], [])
test_eq(_cb.fake_quantizer._act_hooks, {})
test_eq(_cb.frozen, True)   # vacuously: there is no scale to freeze

# observer='dynamic' recomputes the activation scales at run time, and keeps the rounder that does it
_m = _model()
_cb = FakeQuantizeCallback(8, 8, observer='dynamic', freeze_act_pct=0.1)
_fit(_learn(_m), [_cb])
test_eq(fake_quant_spec(_m).observer, 'dynamic')
test_eq([_n for _n, _ in _m.named_buffers() if '_act' in _n], [])
_rounder = next(iter(_cb.fake_quantizer._act_hooks.values()))[1]
test_eq((isinstance(_rounder, _ActRounder), _rounder.dynamic), (True, True))
with torch.no_grad(): _m(torch.randn(2, 3, 8, 8))   # and it still runs, with no frozen scale to read

# a per-layer None leaves one layer out: the only shape where some modules are parametrized and some
# are not, which is what the bake's and the strip's skip branches are for
_m = _model()
_cb = FakeQuantizeCallback(weight_bits=4, act_bits=8, layer_bits={'0': None}, layer_act_bits={'2': None})
_fit(_learn(_m), [_cb])
test_eq('_fp_weight' in _m[0]._buffers, False)    # not rounded, so nothing was snapshotted for it...
test_eq('_act_scale' in _m[0]._buffers, True)     # ...while its activations still are
test_eq('_fp_weight' in _m[2]._buffers, True)
test_eq('_act_scale' in _m[2]._buffers, False)
test_eq((fake_quant_spec(_m).layer_bits, fake_quant_spec(_m).trained), ({'0': None}, True))

# strip() walks the same mixed set, and gives the untouched layer back exactly as it was
_m = _model()
_cb = _install(_m, weight_bits=4, layer_bits={'0': None})
test_eq([_is_parametrized(_m[_i]) for _i in (0, 2, 5)], [False, True, True])
_w0 = _m[0].weight.detach().clone()
_cb.strip()
test_eq(any(_is_parametrized(_mod) for _mod in _m.modules()), False)
test_eq(torch.equal(_m[0].weight.detach(), _w0), True)

## Composition with `Sparsifier`

In [ ]:
# --- composition, order A: both callbacks in one fit (SparsifyCallback runs first, order 0) ---
_m = _bn_model()
_sp_cb = SparsifyCallback(0.5, 'weight', 'local', large_final, one_shot)
_fq_cb = FakeQuantizeCallback(weight_bits=4)     # (Conv2d, Linear), where the Sparsifier takes Conv2d only
_out = _fit(_learn(_m), [_sp_cb, _fq_cb], n=1)

# the mask landed on the master, so the fit really trained a sparse model...
for _conv in (_m[0], _m[3]):
    test_close(_zeros_pct(_conv), 0.5, eps=0.1)
    test_close((_conv.weight == 0).float().mean().item(), 0.5, eps=0.1)   # ...and 0 rounds to 0
# the Linear is rounded but not sparsified, and the BatchNorm is neither: the two layer_types differ
test_eq('_fp_weight' in _m[6]._buffers, True)
test_eq(torch.equal(_m[6].weight.detach(), _fake_quantize(_m[6]._fp_weight, 4, True, 'per_channel')), True)
test_eq('_fp_weight' in _m[1]._buffers, False)
assert not any(_is_parametrized(_mod) for _mod in _m.modules()), 'the fit did not bake'

# `print_sparsity` runs before the bake, on the parametrized model, and reports the MASK: reading the
# rounded weight instead would count the weights the rounding sent to zero as well
_overall = re.search(r'Overall\s+all\s+[\d,]+\s+[\d,]+\s+([\d.]+)%', _out)
assert _overall, _out
test_close(float(_overall.group(1)), 100 * _zeros_pct(_m[0]), eps=2.0)
test_eq(fake_quant_spec(_m).trained, True)

# handing the two callbacks in the other order changes nothing: fastai runs them by `order`, so the
# same fit produces the same model, byte for byte
_m_rev = _bn_model()
_fit(_learn(_m_rev), [FakeQuantizeCallback(weight_bits=4),
                      SparsifyCallback(0.5, 'weight', 'local', large_final, one_shot)], n=1)
_first = dict(_m.named_parameters())
for _n, _p in _m_rev.named_parameters():
    test_eq((_n, torch.equal(_p, _first[_n])), (_n, True))

In [ ]:
# --- composition, order B: a `Sparsifier` built on an ALREADY parametrized model ---
_m = _bn_model()
_cb = _install(_m, weight_bits=4)
_sp = Sparsifier(_m, 'weight', 'local', large_final)   # default layer_type: the convolutions only

# `_save_weights` walks every module with a weight: two parametrized convolutions, a parametrized
# Linear, and a BatchNorm that is not — and it must snapshot the master of each
test_eq(torch.equal(_m[6]._init_weights, _master(_m[6]).detach()), True)
test_ne(_m[6].weight.detach().tolist(), _m[6]._init_weights.tolist())  # the rounding really differs
test_eq(torch.equal(_m[1]._init_weights, _m[1].weight.detach()), True)

_sp.sparsify_model(0.5)
test_close(_zeros_pct(_m[0]), 0.5, eps=0.1)
test_close((_m[0].weight == 0).float().mean().item(), 0.5, eps=0.1)

# the rewind writes the master: a write to `m.weight` would be discarded, and the model would keep
# training from the weights it had
with torch.no_grad(): _master(_m[6]).mul_(0.5)
_sp._reset_weights()
test_eq(torch.equal(_master(_m[6]).detach(), _m[6]._init_weights), True)
test_eq(torch.equal(_master(_m[0]).detach(), _m[0]._init_weights * _m[0]._mask), True)

# a ticket is the floating-point master, saved from a model `torch.save` would otherwise refuse
_ticket_path = os.path.join(tempfile.mkdtemp(), 'ticket.pth')
_sp.save_model(_ticket_path, _m)
_ticket = torch.load(_ticket_path, weights_only=False)
test_eq(_is_parametrized(_ticket[0]), False)
test_eq(torch.equal(_ticket[6].weight.detach(), _m[6]._init_weights), True)
test_eq([_n for _n, _ in _ticket.named_buffers() if _n.endswith(('_mask', '_init_weights'))], [])
test_eq(_ticket(torch.randn(2, 3, 8, 8)).shape, (2, 2))
test_eq(_is_parametrized(_m[0]), True)   # and the live model was not disturbed by the save

In [ ]:
# --- the lottery-ticket path end to end, inside a fit that is also rounding ---
_m = _bn_model()
_sp_cb = SparsifyCallback(0.5, 'weight', 'local', large_final, one_shot, lth=True, save_tickets=True)
_fq_cb = FakeQuantizeCallback(weight_bits=4)
_cwd = os.getcwd()
with tempfile.TemporaryDirectory() as _d:
    os.chdir(_d)
    try:
        _fit(_learn(_m), [_sp_cb, _fq_cb], n=1)
        _tickets = sorted(f for f in os.listdir(_d) if f.startswith('winning_ticket_'))
        _loaded = torch.load(os.path.join(_d, _tickets[-1]), weights_only=False)
    finally:
        os.chdir(_cwd)
assert 'winning_ticket_50.00.pth' in _tickets, _tickets
test_eq(_is_parametrized(_loaded[0]), False)
test_eq(_loaded(torch.randn(2, 3, 8, 8)).shape, (2, 2))
# the ticket is floating point: it is not the rounded weight the fit was training through
test_ne(_loaded[0].weight.detach().tolist(),
        _fake_quantize(_loaded[0].weight.detach(), 4, True, 'per_channel').tolist())

In [ ]:
# --- `Pruner` refuses a parametrized model rather than tracing one ---
_m = _model()
_cb = _install(_m, weight_bits=8)
test_fail(lambda: Pruner(_m, 0.4, 'local', large_final, example_inputs=torch.randn(1, 3, 8, 8)),
          contains='bake or strip it before pruning')
_cb.strip()
Pruner(_m, 0.4, 'local', large_final, example_inputs=torch.randn(1, 3, 8, 8))   # and it works after

## What the documentation may claim

In [ ]:
# --- what the documentation may claim: training through a width is not a saving ---
_FORBIDDEN = ('faster', 'speedup', 'smaller', 'compression', 'shrink')
_docs = {'FakeQuantizeCallback': FakeQuantizeCallback.__doc__,
         **{_n: getattr(FakeQuantizeCallback, _n).__doc__ for _n in ('bake', 'strip', '__init__')}}
for _name, _doc in _docs.items():
    for _word in _FORBIDDEN:   # on word boundaries: the package is called fasterai
        test_eq((_name, _word, bool(re.search(rf'\b{_word}\b', _doc, re.I))), (_name, _word, False))

# the two caveats are the class's own, so `help()` and `show_doc` carry them, not just the page
_class_doc = ' '.join(FakeQuantizeCallback.__doc__.split())   # the docstring wraps; the sentences do not
for _phrase in ('floating-point', 'folds or freezes BatchNorm',
                'before whatever activation function follows it'):
    assert _phrase in _class_doc, _phrase

## Integration (`#| slow`)

In [ ]:
#| slow
# --- integration: a real architecture, trained through a width, on a real `DataLoaders` ---
from torchvision.models import resnet18

torch.manual_seed(0)
_xr = torch.randn(16, 3, 32, 32)
_dls_r = DataLoaders.from_dsets(TensorDataset(_xr, torch.randint(0, 10, (16,))),
                                TensorDataset(_xr, torch.randint(0, 10, (16,))), bs=8, device='cpu')
_rn = resnet18(weights=None, num_classes=10)
_w0 = _rn.layer4[1].conv2.weight.detach().clone()
_keys = list(_rn.state_dict())
_learn_r = Learner(_dls_r, _rn, loss_func=nn.CrossEntropyLoss())
_cb = FakeQuantizeCallback(8, 8, freeze_act_pct=0.5)
_loss_fp = _learn_r.validate()[0]
_out = _fit(_learn_r, [_cb], n=1)

test_eq(sorted(_rn.state_dict()), sorted(_keys))
assert not any(_is_parametrized(_m) for _m in _rn.modules()), 'a parametrization survived the bake'
test_eq(fake_quant_spec(_rn).trained, True)
test_eq(_cb.frozen, True)
# the BatchNorm layers are untouched: not folded, not frozen, and still training their statistics
test_eq(_rn.bn1.training, False)   # `validate` below leaves the model in eval
test_ne(_rn.layer4[1].conv2.weight.detach().tolist(), _w0.tolist())
_loss_q = _learn_r.validate()[0]
assert torch.isfinite(torch.tensor(_loss_q)), _loss_q
print(f'fp32 {_loss_fp:.4f}  after W8A8 QAT {_loss_q:.4f}')

# every convolution holds the rounding of its own trained master
for _name in ('conv1', 'layer1.0.conv1', 'layer4.1.conv2'):
    _mod = _rn.get_submodule(_name)
    test_eq((_name, torch.equal(_mod.weight.detach(),
                                _fake_quantize(_mod._fp_weight, 8, True, 'per_channel'))), (_name, True))
_cb.fake_quantizer.remove()
test_eq([_n for _n, _ in _rn.named_buffers() if '_fp_weight' in _n or '_act_' in _n], [])

In [ ]:
#| slow
# --- integration: sparsify and round the same ResNet, in one fit ---
torch.manual_seed(0)
_rn = resnet18(weights=None, num_classes=10)
_learn_rs = Learner(_dls_r, _rn, loss_func=nn.CrossEntropyLoss())
_out = _fit(_learn_rs, [SparsifyCallback(0.5, 'weight', 'local', large_final, one_shot),
                        FakeQuantizeCallback(weight_bits=8)], n=1)
_conv = _rn.layer1[0].conv1
test_close((_conv.weight == 0).float().mean().item(), 0.5, eps=0.1)
test_eq(fake_quant_spec(_rn).trained, True)
assert not any(_is_parametrized(_m) for _m in _rn.modules())
with torch.no_grad(): assert torch.isfinite(_rn(_xr)).all()